# Multi-Engine LLM Serving Benchmark — Cloud Track (NVIDIA Tesla T4 16GB)
**Engines:** `vLLM` (PagedAttention) vs. `SGLang` (RadixAttention)  
**Model:** `google/gemma-4-E4B-it` (`FP16`, `INT8`)  
**Concurrency Ladder:** `1 / 8 / 32 / 128` + **Prefix Caching Ablation**

> **IMPORTANT:** When selecting your Colab runtime variant in VS Code, select **`GPU` (T4)** — **NOT** `TPU` or `CPU`.

In [11]:
# Cell 1: Verify you are connected to an NVIDIA Tesla T4 GPU (16GB VRAM)
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# Cell 2: Install vLLM, SGLang, bitsandbytes, and httpx on the remote Colab VM
!pip install -q -U vllm "sglang[all]" bitsandbytes httpx

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=

In [3]:
# Cell 3: Bootstrap the Benchmark Harness onto the remote Colab VM
# (Required when using the VS Code Colab extension so remote cells have all local modules)
import os, pathlib
pathlib.Path('harness').mkdir(exist_ok=True)
pathlib.Path('scripts/colab').mkdir(parents=True, exist_ok=True)
pathlib.Path('results').mkdir(exist_ok=True)
pathlib.Path('harness/__init__.py').write_text('')
print('Remote directories initialized in:', os.getcwd())

Remote directories initialized in: /content


In [4]:
%%writefile harness/schema.py
from __future__ import annotations
from dataclasses import asdict, dataclass, field
import math, platform, statistics
from typing import Any

@dataclass(frozen=True)
class RequestTelemetry:
    request_id: int
    success: bool
    prompt_tokens_est: int
    output_tokens: int
    ttft_ms: float
    e2e_latency_ms: float
    tpot_ms: float
    itl_ms_list: list[float] = field(default_factory=list)
    error_message: str | None = None

@dataclass(frozen=True)
class DistributionSummary:
    mean_ms: float
    p50_ms: float
    p90_ms: float
    p99_ms: float
    min_ms: float
    max_ms: float

@dataclass(frozen=True)
class BenchmarkCellResult:
    timestamp_utc: str
    hardware_target: str
    os_platform: str
    engine: str
    model_id: str
    quantization: str
    workload_mode: str
    concurrency: int
    total_requests: int
    successful_requests: int
    failed_requests: int
    wall_time_s: float
    total_output_tokens: int
    system_output_throughput_tps: float
    mean_per_user_tps: float
    ttft: DistributionSummary
    tpot: DistributionSummary
    itl: DistributionSummary
    e2e: DistributionSummary
    rss_memory_mb: float

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

def compute_percentile(sorted_values: list[float], percentile: float) -> float:
    if not sorted_values:
        return 0.0
    if len(sorted_values) == 1:
        return round(sorted_values[0], 3)
    rank = (percentile / 100.0) * (len(sorted_values) - 1)
    lo, hi = int(math.floor(rank)), int(math.ceil(rank))
    w = rank - lo
    return round(sorted_values[lo] * (1.0 - w) + sorted_values[hi] * w, 3)

def summarize_distribution(values: list[float]) -> DistributionSummary:
    if not values:
        return DistributionSummary(0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
    ordered = sorted(values)
    return DistributionSummary(
        mean_ms=round(statistics.fmean(ordered), 3),
        p50_ms=compute_percentile(ordered, 50.0),
        p90_ms=compute_percentile(ordered, 90.0),
        p99_ms=compute_percentile(ordered, 99.0),
        min_ms=round(ordered[0], 3),
        max_ms=round(ordered[-1], 3),
    )

def detect_host_fingerprint() -> str:
    return f"{platform.system()}-{platform.machine()}-{platform.processor() or 'unknown'}"

Writing harness/schema.py


In [5]:
%%writefile harness/prompts.py
from __future__ import annotations

SHARED_SYSTEM_CONTEXT = (
    "You are a Principal Distributed Systems Architect evaluating high-throughput "
    "inference clusters. Reference architecture parameters:\n"
    "1. Compute Node Topology: Dual-tier heterogeneous cluster combining Apple Silicon "
    "M-series unified memory nodes (100 GB/s LPDDR5-6400 bus, 16 GB capacity) and "
    "NVIDIA Turing T4 GDDR6 nodes (320 GB/s memory bus, 16 GB capacity, 65 TFLOPS FP16).\n"
    "2. Execution Phases: Prefill computes parallel key-value projections across all "
    "input tokens (compute-bound GEMM), whereas Decode computes autoregressive token "
    "generation one step at a time (memory-bandwidth-bound GEMV at low batch sizes).\n"
    "3. KV Cache Mechanics: Standard contiguous pre-allocation fragments up to 60% of "
    "VRAM. PagedAttention partitions KV states into fixed-size non-contiguous blocks, "
    "eliminating external fragmentation. RadixAttention maintains a radix tree of "
    "completed KV blocks across requests to enable O(1) prefix reuse.\n"
    "4. Quantization Tradeoffs: Reducing weight representation from 16-bit floating "
    "point (2 bytes/param) to 8-bit integer (1 byte/param) or 4-bit group-quantized "
    "formats (0.5 bytes/param) proportionally reduces memory bus transfer time during "
    "decode at the cost of dequantization kernel overhead during high-batch compute.\n"
    "5. Service Level Objectives: P99 Time-To-First-Token (TTFT) must remain below "
    "1,500 milliseconds and P90 Time-Per-Output-Token (TPOT) must remain below 50 "
    "milliseconds under sustained multi-tenant load.\n"
) * 3

UNIQUE_TOPICS = [
    "continuous batching iteration-level scheduling",
    "speculative decoding draft verification acceptance rates",
    "tensor parallelism all-reduce communication overhead",
    "pipeline parallelism bubble minimization micro-batches",
    "quantization-aware training versus post-training AWQ",
    "KV cache quantization to FP8 and INT4 formats",
    "chunked prefill head-of-line blocking prevention",
    "flash attention SRAM tiling and online softmax",
]

def build_messages(request_id: int, workload_mode: str) -> list[dict[str, str]]:
    topic = UNIQUE_TOPICS[request_id % len(UNIQUE_TOPICS)]
    if workload_mode == "prefix_cache":
        return [
            {"role": "system", "content": SHARED_SYSTEM_CONTEXT},
            {"role": "user", "content": f"Request #{request_id}: Analyze {topic} covering hardware bottlenecks and latency math in 300 words."},
        ]
    return [
        {"role": "user", "content": f"[Unique-Session-ID-{request_id:05d}-Seed-{request_id * 7919}] Provide a first-principles breakdown of {topic} in 300 words."}
    ]

Writing harness/prompts.py


In [6]:
%%writefile harness/benchmark.py
from __future__ import annotations
import asyncio, csv, json, resource, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import httpx
from harness.prompts import build_messages
from harness.schema import BenchmarkCellResult, RequestTelemetry, detect_host_fingerprint, summarize_distribution

async def measure_single_stream(client: httpx.AsyncClient, endpoint_url: str, model_id: str, request_id: int, workload_mode: str, max_tokens: int) -> RequestTelemetry:
    messages = build_messages(request_id=request_id, workload_mode=workload_mode)
    prompt_tokens_est = max(1, sum(len(m["content"]) for m in messages) // 4)
    payload: dict[str, Any] = {"model": model_id, "messages": messages, "max_tokens": max_tokens, "temperature": 0.0, "stream": True}
    token_timestamps_ns: list[int] = []
    start_ns = time.perf_counter_ns()
    try:
        async with client.stream("POST", endpoint_url, json=payload) as response:
            response.raise_for_status()
            async for raw_line in response.aiter_lines():
                line = raw_line.strip()
                if not line or not line.startswith("data:"):
                    continue
                data_str = line[5:].strip()
                if data_str == "[DONE]":
                    break
                chunk = json.loads(data_str)
                choices = chunk.get("choices") or []
                if not choices:
                    continue
                delta = choices[0].get("delta") or {}
                if delta.get("content") or delta.get("reasoning_content"):
                    token_timestamps_ns.append(time.perf_counter_ns())
    except Exception as exc:
        return RequestTelemetry(request_id, False, prompt_tokens_est, 0, 0.0, round((time.perf_counter_ns() - start_ns) / 1e6, 3), 0.0, error_message=str(exc))

    end_ns = time.perf_counter_ns()
    n_tok = len(token_timestamps_ns)
    if n_tok == 0:
        return RequestTelemetry(request_id, False, prompt_tokens_est, 0, 0.0, round((end_ns - start_ns) / 1e6, 3), 0.0, error_message="Empty stream")
    ttft_ms = (token_timestamps_ns[0] - start_ns) / 1e6
    e2e_ms = (end_ns - start_ns) / 1e6
    itls = [round((token_timestamps_ns[i] - token_timestamps_ns[i - 1]) / 1e6, 3) for i in range(1, n_tok)]
    tpot_ms = ((token_timestamps_ns[-1] - token_timestamps_ns[0]) / 1e6) / (n_tok - 1) if n_tok > 1 else 0.0
    return RequestTelemetry(request_id, True, prompt_tokens_est, n_tok, round(ttft_ms, 3), round(e2e_ms, 3), round(tpot_ms, 3), itls)

async def run_benchmark_cell(base_url: str, engine: str, model_id: str, quantization: str, hardware_target: str, concurrency: int, total_requests: int, max_tokens: int, workload_mode: str = "standard", warmup_requests: int = 2) -> BenchmarkCellResult:
    endpoint_url = f"{base_url.rstrip('/')}/v1/chat/completions"
    limits = httpx.Limits(max_connections=concurrency + 16, max_keepalive_connections=concurrency)
    timeout = httpx.Timeout(connect=30.0, read=300.0, write=30.0, pool=300.0)
    async with httpx.AsyncClient(limits=limits, timeout=timeout) as client:
        for w_idx in range(warmup_requests):
            await measure_single_stream(client, endpoint_url, model_id, 9000 + w_idx, workload_mode, min(32, max_tokens))
        sem = asyncio.Semaphore(concurrency)
        async def _worker(i: int) -> RequestTelemetry:
            async with sem:
                return await measure_single_stream(client, endpoint_url, model_id, i, workload_mode, max_tokens)
        t0 = time.perf_counter_ns()
        records = await asyncio.gather(*[_worker(i) for i in range(total_requests)])
        wall_s = (time.perf_counter_ns() - t0) / 1e9

    ok = [r for r in records if r.success]
    all_itls = [x for r in ok for x in r.itl_ms_list]
    tot_tok = sum(r.output_tokens for r in ok)
    sys_tps = tot_tok / wall_s if wall_s > 0 else 0.0
    u_tps = [1000.0 / r.tpot_ms for r in ok if r.tpot_ms > 0]
    return BenchmarkCellResult(
        timestamp_utc=datetime.now(timezone.utc).isoformat(),
        hardware_target=hardware_target,
        os_platform=detect_host_fingerprint(),
        engine=engine, model_id=model_id, quantization=quantization,
        workload_mode=workload_mode, concurrency=concurrency,
        total_requests=total_requests, successful_requests=len(ok),
        failed_requests=len(records) - len(ok), wall_time_s=round(wall_s, 3),
        total_output_tokens=tot_tok, system_output_throughput_tps=round(sys_tps, 2),
        mean_per_user_tps=round(sum(u_tps) / len(u_tps), 2) if u_tps else 0.0,
        ttft=summarize_distribution([r.ttft_ms for r in ok]),
        tpot=summarize_distribution([r.tpot_ms for r in ok if r.tpot_ms > 0]),
        itl=summarize_distribution(all_itls),
        e2e=summarize_distribution([r.e2e_latency_ms for r in ok]),
        rss_memory_mb=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0, 2),
    )

def persist_cell_result(result: BenchmarkCellResult, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    with (output_dir / "benchmark_results.jsonl").open("a", encoding="utf-8") as jf:
        jf.write(json.dumps(result.to_dict()) + "\n")
    flat = {
        "timestamp_utc": result.timestamp_utc, "hardware_target": result.hardware_target,
        "engine": result.engine, "model_id": result.model_id, "quantization": result.quantization,
        "workload_mode": result.workload_mode, "concurrency": result.concurrency,
        "ok_requests": result.successful_requests, "failed_requests": result.failed_requests,
        "system_throughput_tps": result.system_output_throughput_tps,
        "mean_per_user_tps": result.mean_per_user_tps,
        "ttft_p50_ms": result.ttft.p50_ms, "ttft_p90_ms": result.ttft.p90_ms, "ttft_p99_ms": result.ttft.p99_ms,
        "tpot_p50_ms": result.tpot.p50_ms, "tpot_p90_ms": result.tpot.p90_ms, "tpot_p99_ms": result.tpot.p99_ms,
        "e2e_p50_ms": result.e2e.p50_ms, "e2e_p99_ms": result.e2e.p99_ms,
    }
    csv_path = output_dir / "benchmark_summary.csv"
    write_hdr = not csv_path.exists()
    with csv_path.open("a", newline="", encoding="utf-8") as cf:
        w = csv.DictWriter(cf, fieldnames=list(flat.keys()))
        if write_hdr:
            w.writeheader()
        w.writerow(flat)

Writing harness/benchmark.py


In [7]:
%%writefile scripts/colab/run_t4_suite.py
from __future__ import annotations
import argparse, asyncio, subprocess, sys, time
from contextlib import asynccontextmanager
from pathlib import Path
import httpx
sys.path.insert(0, str(Path.cwd()))
from harness.benchmark import persist_cell_result, run_benchmark_cell

def query_nvidia_vram_used_mb() -> float:
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"], text=True)
        return float(out.strip().splitlines()[0])
    except Exception:
        return 0.0

def build_cuda_engine_command(engine: str, model_repo: str, quantization: str, port: int, max_model_len: int = 2048, gpu_mem_fraction: float = 0.88) -> list[str]:
    if engine == "vllm":
        cmd = [
            sys.executable, "-m", "vllm.entrypoints.openai.api_server",
            "--model", model_repo, "--host", "127.0.0.1", "--port", str(port),
            "--dtype", "float16", "--max-model-len", str(max_model_len),
            "--gpu-memory-utilization", str(gpu_mem_fraction),
            "--enforce-eager",
            "--limit-mm-per-prompt", '{"image":0,"video":0,"audio":0}',
            "--enable-prefix-caching", "--disable-log-requests",
        ]
        if quantization == "INT8":
            cmd.extend(["--quantization", "int8_per_channel_weight_only"])
        return cmd
    if engine == "sglang":
        cmd = [
            sys.executable, "-m", "sglang.launch_server",
            "--model-path", model_repo, "--host", "127.0.0.1", "--port", str(port),
            "--dtype", "float16", "--context-length", str(max_model_len),
            "--mem-fraction-static", "0.82", "--attention-backend", "triton",
            "--disable-cuda-graph",
        ]
        if quantization == "INT8":
            cmd.extend(["--quantization", "w8a8_int8"])
        return cmd
    raise ValueError(engine)

def cleanup_gpu_processes() -> None:
    subprocess.run(["pkill", "-9", "-f", "vllm.entrypoints|sglang.launch_server|VLLM::EngineCore|sglang::"], check=False)
    time.sleep(2.0)

@asynccontextmanager
async def managed_cuda_server(cmd: list[str], base_url: str, startup_timeout_s: float = 600.0):
    cleanup_gpu_processes()
    log_path = Path("cuda_server.log")
    log_file = log_path.open("w", encoding="utf-8")
    print(f"\n[CUDA Server] Free VRAM before launch — Used: {query_nvidia_vram_used_mb():.0f} MiB")
    print(f"[CUDA Server] Launching: {' '.join(cmd)}")
    proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    deadline = time.monotonic() + startup_timeout_s
    ready = False
    step = 0
    try:
        async with httpx.AsyncClient(timeout=3.0) as client:
            while time.monotonic() < deadline:
                if proc.poll() is not None:
                    log_file.flush()
                    logs = log_path.read_text(encoding="utf-8", errors="replace")
                    raise RuntimeError(f"CUDA server exited (code {proc.returncode}):\n{logs[-6000:]}")
                try:
                    resp = await client.get(f"{base_url}/v1/models")
                    if resp.status_code == 200:
                        ready = True
                        break
                except httpx.HTTPError:
                    pass
                step += 1
                if step % 6 == 0:
                    log_file.flush()
                    lines = [ln.strip() for ln in log_path.read_text(encoding="utf-8", errors="replace").splitlines() if ln.strip()]
                    last_ln = lines[-1][-100:] if lines else "starting..."
                    print(f"  [Booting {step*2}s] VRAM: {query_nvidia_vram_used_mb():.0f} MiB | {last_ln}")
                await asyncio.sleep(2.0)
        if not ready:
            logs = log_path.read_text(encoding="utf-8", errors="replace")
            raise TimeoutError(f"Server startup timed out. Last logs:\n{logs[-4000:]}")
        print(f"[CUDA Server] Ready | Active VRAM: {query_nvidia_vram_used_mb():.0f} MiB")
        yield proc
    finally:
        log_file.close()
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10.0)
            except subprocess.TimeoutExpired:
                proc.kill()
        cleanup_gpu_processes()

async def run_t4_sweep(engine: str, model_repo: str, model_label: str, quantization: str, concurrencies: list[int], max_tokens: int, workload_mode: str, port: int, output_dir: Path) -> None:
    base_url = f"http://127.0.0.1:{port}"
    cmd = build_cuda_engine_command(engine, model_repo, quantization, port)
    async with managed_cuda_server(cmd, base_url):
        for c in concurrencies:
            total_reqs = max(4, c * 2)
            print(f"  -> Cell: engine={engine} quant={quantization} workload={workload_mode} C={c} reqs={total_reqs}")
            res = await run_benchmark_cell(base_url, engine, model_label, quantization, "NVIDIA-T4-16GB", c, total_reqs, max_tokens, workload_mode, warmup_requests=2)
            persist_cell_result(res, output_dir)
            print(f"     [Done] Sys Throughput: {res.system_output_throughput_tps:6.2f} tok/s | User Speed: {res.mean_per_user_tps:5.2f} tok/s | TTFT P50: {res.ttft.p50_ms:6.1f} ms | TPOT P50: {res.tpot.p50_ms:5.1f} ms | VRAM: {query_nvidia_vram_used_mb():.0f} MiB")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--engine", choices=["vllm", "sglang"], required=True)
    p.add_argument("--model-repo", required=True)
    p.add_argument("--model-label", default="google/gemma-4-E4B-it")
    p.add_argument("--quant", choices=["FP16", "INT8", "Q4_K_M"], required=True)
    p.add_argument("--concurrencies", default="1,8,32")
    p.add_argument("--max-tokens", type=int, default=128)
    p.add_argument("--workload", choices=["standard", "prefix_cache"], default="standard")
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--output-dir", default="results")
    a = p.parse_args()
    asyncio.run(run_t4_sweep(a.engine, a.model_repo, a.model_label, a.quant, [int(x) for x in a.concurrencies.split(",")], a.max_tokens, a.workload, a.port, Path(a.output_dir)))

Writing scripts/colab/run_t4_suite.py


In [8]:
# Cell 7: Run vLLM Benchmark on Pre-Quantized 4-bit Gemma 4 E4B (Uses only ~4.5 GB VRAM!)
!python3 scripts/colab/run_t4_suite.py \
  --engine vllm \
  --model-repo axteam/gemma-4-E4B-it-AWQ-4bit \
  --quant Q4_K_M \
  --concurrencies "1,8,32" \
  --workload standard

!python3 scripts/colab/run_t4_suite.py \
  --engine vllm \
  --model-repo axteam/gemma-4-E4B-it-AWQ-4bit \
  --quant Q4_K_M \
  --concurrencies "1,8,32" \
  --workload prefix_cache


[CUDA Server] Free VRAM before launch — Used: 0 MiB
[CUDA Server] Launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model axteam/gemma-4-E4B-it-AWQ-4bit --host 127.0.0.1 --port 8000 --dtype float16 --max-model-len 2048 --gpu-memory-utilization 0.88 --enforce-eager --limit-mm-per-prompt {"image":0,"video":0,"audio":0} --enable-prefix-caching --disable-log-requests
  [Booting 12s] VRAM: 0 MiB | starting...
  [Booting 24s] VRAM: 0 MiB | 3000 9965 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
  [Booting 36s] VRAM: 0 MiB | 3000 9965 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
Traceback (most recent call last):
  File "/content/scripts/colab/run_t4_suite.py", line 117, in <module>
    asyncio.run(run_t4_sweep(a.engine, a.model_repo, a.model_label, a.quant, [int(x) for x in a.concurrencies.split(",")], a.max_tokens, a.workload, a.port, Path(a.output_dir)))
    ~~~~~~~~~~~^^^^^^^^^^^^

In [9]:
# Cell 8: Run SGLang Benchmark on Pre-Quantized 4-bit Gemma 4 E4B + Prefix Caching Showdown
!python3 scripts/colab/run_t4_suite.py \
  --engine sglang \
  --model-repo axteam/gemma-4-E4B-it-AWQ-4bit \
  --quant Q4_K_M \
  --concurrencies "1,8,32" \
  --workload standard

!python3 scripts/colab/run_t4_suite.py \
  --engine sglang \
  --model-repo axteam/gemma-4-E4B-it-AWQ-4bit \
  --quant Q4_K_M \
  --concurrencies "1,8,32" \
  --workload prefix_cache


[CUDA Server] Free VRAM before launch — Used: 0 MiB
[CUDA Server] Launching: /usr/bin/python3 -m sglang.launch_server --model-path axteam/gemma-4-E4B-it-AWQ-4bit --host 127.0.0.1 --port 8000 --dtype float16 --context-length 2048 --mem-fraction-static 0.82 --attention-backend triton --disable-cuda-graph
  [Booting 12s] VRAM: 0 MiB | starting...
Traceback (most recent call last):
  File "/content/scripts/colab/run_t4_suite.py", line 117, in <module>
    asyncio.run(run_t4_sweep(a.engine, a.model_repo, a.model_label, a.quant, [int(x) for x in a.concurrencies.split(",")], a.max_tokens, a.workload, a.port, Path(a.output_dir)))
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
   

In [10]:
# Cell 9: Print the CSV Summary so we can copy/merge it into your Mac's results/benchmark_summary.csv!
!cat results/benchmark_summary.csv

cat: results/benchmark_summary.csv: No such file or directory
